In [2]:
import pandas as pd
import folium
import geopandas as gpd
from shapely.geometry import Point, LineString
from IPython.display import display
import numpy as np
import webbrowser

In [4]:


# Ask user which voltage levels they want to plot
plot_high = True
plot_medium = True
plot_low = True

# Choose to move units to the nearest bus or not
move_gens_to_nearest_bus = True

# define the color for each type
gen_colors = ['green',   'orange', 'blue', 'black','purple','red', 'cyan', 'magenta', 'brown']

# extract the unique types of generation units
generation_types = REE_tech_categoroes = ['Wind', 'Solar', 'Hydro',  'Coal',"Nuclear", 'Combined Cycle', 'Gas Turbine', 'Steam Turbine', 'Other Renewables']

# Create a dictionary that maps each generation type to a color
color_map = dict(zip(generation_types, gen_colors))

technology_to_resource_map = {
    "Offshore floating": "Wind",
    "Onshore": "Wind",
    "Coal": "Coal",
    "Solar Thermal": "Solar",
    "PV": "Solar",
    "run_of_river": "Hydro",
    "reservoir": "Hydro",
    "pumped_storage": "Hydro",
    "Nuclear": "Nuclear",
    "Combined_cycle": "Combined Cycle",
    "Gas_turbine": "Gas Turbine",
    "Steam_turbine": "Steam Turbine",
    "ICCC": "Other Renewables",
    "Biomass": "Other Renewables"
}

boolean_gen_type_plot ={
    "Offshore floating": True,
    "Onshore": True,
    "Coal": True,
    "Solar Thermal": True,
    "PV": True,
    "run_of_river": True,
    "reservoir": True,
    "pumped_storage": True,
    "Nuclear": True,
    "Combined_cycle": True,
    "Gas_turbine": True,
    "Steam_turbine": True,
    "ICCC": True,
    "Biomass": True
}

#  file locations
buses_file = 'Bus_Data.csv'
# lines_file = 'lines.csv'
lines_file = 'C:\\Users\\ehsanno\\DataspellProjects\\IAMC_format\\totally_merged_parallel_lines.csv'
generation_file = 'generation.csv'
gadm_shapefile = 'M:/Work/iDesignRes/Spain/spain_shape/es.shp'


# reading the file
try:
    generation_ree = pd.read_csv(generation_file)
    buses_relevant = pd.read_csv(buses_file)
    lines_relevant = pd.read_csv(lines_file)
except Exception as e:
    print(f"Error loading files: {e}")


# Define color mappings for voltage levels
voltage_colors = {
    'high': 'black',
    'medium': 'red',
    'low': 'green',
    'outside': 'yellow'
}

# Load the GADM shapefile and filter for Spain
spain = gpd.read_file(gadm_shapefile)

# Ensure the CRS is the same for both datasets
spain = spain.to_crs(epsg=4326)

# Create a folium map centered around Spain with a specific width and height
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, width='100%', height='80%')

# Add Spain border to the map with a specified border color and opacity
folium.GeoJson(
    spain.geometry,
    name="Spain Border",
    style_function=lambda x: {'color': 'gray', 'weight': 0.01, 'opacity': 0.5}  # Adjust the color, weight, and opacity as needed
).add_to(m)

# Convert buses DataFrame to GeoDataFrame
buses_gdf = gpd.GeoDataFrame(buses_relevant, geometry=gpd.points_from_xy(buses_relevant.x, buses_relevant.y), crs="EPSG:4326")

# Plot the buses with different colors based on voltage level
for idx, bus in buses_gdf.iterrows():
    if bus['country'] == 'ES':
        voltage_level = bus['voltage']  # Adjust this if your column name is different
        if voltage_level >= 300 and plot_high:
            color = voltage_colors['high']
        elif 200 <= voltage_level < 300 and plot_medium:
            color = voltage_colors['medium']
        elif voltage_level < 200 and plot_low:
            color = voltage_colors['low']
        else:
            continue  # Skip this bus if the voltage level is not selected
    else:
        color = voltage_colors['outside']

    radius = 4 if color == voltage_colors['outside'] else 2 if color == voltage_colors['high'] else 1

    folium.CircleMarker(
        location=[bus.geometry.y, bus.geometry.x],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=1,
        popup=folium.Popup(f"Bus ID: {bus['bus_id']}<br>Voltage: {bus['voltage']}", max_width=300)
    ).add_to(m)

    # Plot the lines by connecting buses based on bus_id from lines data
for idx, line in lines_relevant.iterrows():
    bus0 = buses_relevant[buses_relevant['bus_id'] == line['bus0']]
    bus1 = buses_relevant[buses_relevant['bus_id'] == line['bus1']]

    if not bus0.empty and not bus1.empty:
        x_values = [bus0.x.values[0], bus1.x.values[0]]
        y_values = [bus0.y.values[0], bus1.y.values[0]]


        voltage_level = line['voltage']  # Adjust this if your column name is different
        if voltage_level >= 300 and plot_high:
            color = voltage_colors['high']
            weight = 2.5
        elif 200 <= voltage_level < 300 and plot_medium:
            color = voltage_colors['medium']
            weight = 1
        elif voltage_level < 200 and plot_low:
            color = voltage_colors['low']
            weight = 1
        else:
            continue  # Skip this line if the voltage level is not selected

        folium.PolyLine(
            locations=[[y_values[0], x_values[0]], [y_values[1], x_values[1]]],
            color=color,
            weight=weight
        ).add_to(m)



# Calculate and print the number of buses at each voltage level
voltage_counts = buses_relevant['voltage'].value_counts().sort_index()
print("Number of buses at each voltage level:")
print(voltage_counts)

# Generation Units


if move_gens_to_nearest_bus:
    for idx, gen in generation_ree.iterrows():
        if boolean_gen_type_plot[gen['technology']]:
            folium.CircleMarker(
                location=[gen.y_bus, gen.x_bus],
                radius=np.sqrt(gen['capacity_mw'])/1.5,  # Use square root to scale the area proportionally to the capacity
                color=color_map[technology_to_resource_map[gen['technology']]],
                fill=True,
                fill_color=color_map[technology_to_resource_map[gen['technology']]],
                fill_opacity=0.5,
                stroke=False,  # Remove the stroke to eliminate the lines around the circles
                popup=folium.Popup(f"Generator ID: {gen['name']}<br>Capacity: {gen['capacity_mw']} MW<br>Type: {gen['technology']}", max_width=300)
            ).add_to(m)
        
else:
    for idx, gen in generation_ree.iterrows():
        if boolean_gen_type_plot[gen['technology']]:
            folium.CircleMarker(
                location=[gen.y, gen.x],
                radius=np.sqrt(gen['capacity_mw'])/1.5,  # Use square root to scale the area proportionally to the capacity
                color=color_map[technology_to_resource_map[gen['technology']]],
                fill=True,
                fill_color=color_map[technology_to_resource_map[gen['technology']]],
                fill_opacity=0.5,
                stroke=False,  # Remove the stroke to eliminate the lines around the circles
                popup=folium.Popup(f"Generator ID: {gen['name']}<br>Capacity: {gen['capacity_mw']} MW<br>Type: {gen['technology']}", max_width=300)
            ).add_to(m)
        
            
map_file = 'spain_power_network_map.html'
m.save(map_file)

# Open the map in the default web browser
webbrowser.open(map_file)
    

Number of buses at each voltage level:
voltage
132     39
220    867
225      4
250      2
320      3
400    321
Name: count, dtype: int64


True

In [6]:
color_map

{'Wind': 'green',
 'Solar': 'orange',
 'Hydro': 'blue',
 'Coal': 'black',
 'Nuclear': 'purple',
 'Combined Cycle': 'red',
 'Gas Turbine': 'cyan',
 'Steam Turbine': 'magenta',
 'Other Renewables': 'brown'}

'Biomass'